# tools.search

> Web search as a tool the model can call, backed by the Brave Search API.

In [ ]:
#| default_exp tools.search

In [ ]:
#| hide
from nbdev.showdoc import *

## The tool

Brave's REST API returns a JSON document; we keep only the top `k` web results and collapse each into a short `title / url / description` block. That gives the model enough to decide whether to follow up (potentially with another tool, like a future `fetch_url`) without spending tokens on Brave's auxiliary fields.

The schema is hand-written for now. Auto-generating it from the function's signature is a fine future move, but writing it explicitly here keeps the dependency surface tiny and makes the contract obvious to a reader of the file.

In [ ]:
#| export
import os
import httpx
from nbdialog.core import Tool

BRAVE_SEARCH_URL = "https://api.search.brave.com/res/v1/web/search"

def _brave_search(query: str, k: int = 5, api_key_env: str = "BRAVE_API_KEY") -> str:
    "Call the Brave Search API and return a compact markdown digest of the top `k` web results."
    r = httpx.get(
        BRAVE_SEARCH_URL,
        params={"q": query, "count": k},
        headers={"X-Subscription-Token": os.environ[api_key_env],
                 "Accept": "application/json"},
        timeout=20.0,
    )
    r.raise_for_status()
    results = (r.json().get("web") or {}).get("results", [])[:k]
    if not results: return "(no results)"
    return "\n\n".join(
        f"## {x.get('title','').strip()}\n{x.get('url','')}\n{x.get('description','').strip()}"
        for x in results
    )

web_search = Tool(
    schema={
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the public web via the Brave Search API. Returns a short markdown digest of the top results (title, URL, snippet) so the model can decide which to follow up on.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query."},
                    "k": {"type": "integer", "description": "How many results to return (default 5, max 20).", "default": 5},
                },
                "required": ["query"],
            },
        },
    },
    fn=_brave_search,
)

Smoke test — only runs when the key is present, so doc builds without credentials stay green:

In [ ]:
if os.environ.get("BRAVE_API_KEY"):
    print(_brave_search("nbdev fastai", k=2)[:400])

## nbdev – Create delightful software with Jupyter Notebooks ...
https://nbdev.fast.ai/
Write, test, document, and distribute software packages and technical articles — all in one place, your notebook.

## End-To-End Walkthrough - nbdev - Fast.ai
https://nbdev.fast.ai/tutorial.html
Write, test, document, and distribute software packages and technical articles — all in one place, your notebook.


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()